# 08 — Export train / valid / test as CSV, with the top-10 of the main model and baselines

Main model = SMLP4Rec + query embedding (trained with the first bookings, notebook 01d) + sameDest (weights tuned on valid), reference seed 2022. Same checkpoint, vocabulary and sameDest weights as `scripts/demo_ui.py`.
Baselines = ItemKNN and logistic regression of notebook 07b (same training rows as the main model), loaded via `demo_ui.Baselines`.

- **Split**: global temporal 80 / 10 / 10 over next-booking targets, as in 01c / 01d. The boundaries (`first_target_event`) come from the run json, not retyped. Event id = line index of the booking in `expedia_bookings_no_tgt.csv` (identical to the RecBole event index).
- **Train rows**: every booking up to the last train target, exactly as used to fit the models. First bookings (`pos = 0`) are kept.
- **Valid and test extra**: `top10_clusters` (main model), `top10_clusters_itemknn` (ItemKNN), and `top10_clusters_lr` (logistic regression) (space-separated, best first). The recall@10 in the counts json is computed from the full ranking before the helper columns are dropped.
- **Leak prevention**: all `tgt_*` columns and `aux_cnt` were stripped at export time; verified by assert below.
- Cluster ids are anonymous (proxy of room category / package). Offline only.
- Valid rows were used to tune sameDest and pick the epoch, so valid top-10 is not an unbiased score. Output: `data/processed/splits_csv/{train,valid,test}.csv` (gitignored) and counts in `results/week4_rebuild/split_export_counts.json`.

Kernel: `smlp4rec` (Python 3.11).


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch  # before pandas: on Windows, pandas first makes c10.dll fail to load

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src"), str(ROOT / "scripts")]

import pandas as pd

import demo_ui as demo  # Data, Runner: the same loading code as the demo
from nbp.data.query_features import QUERY_FIELDS, encode
from nbp.hybrid.fusion import same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores

SEED = 2022
TOP_K = 10
BATCH = 4096
OUT = ROOT / "data" / "processed" / "splits_csv"
OUT.mkdir(parents=True, exist_ok=True)
COUNTS_JSON = ROOT / "results" / "week4_rebuild" / "split_export_counts.json"

In [ ]:
data = demo.Data()
runner = demo.Runner(SEED, data)
baselines = demo.Baselines()
folder, js = demo.RUNS[SEED]
meta = json.loads((ROOT / "results" / "week4_rebuild" / js).read_text(encoding="utf-8"))
sp = meta["splits"]
bounds = {k: (sp[k]["first_target_event"], sp[k]["last_target_event"]) for k in ("train", "valid", "test")}
n_events = len(data.user)
assert bounds["test"][1] == n_events - 1 and bounds["valid"][1] + 1 == bounds["test"][0]
split = np.empty(n_events, dtype=object)
split[: bounds["valid"][0]] = "train"  # includes the first events before the first target
split[bounds["valid"][0] : bounds["test"][0]] = "valid"
split[bounds["test"][0] :] = "test"
print(bounds, pd.Series(split).value_counts().to_dict())


In [ ]:
# the smlp4rec venv has no pyarrow: the project .venv (pandas 3) converts the parquet to a csv once
BK_CSV = ROOT / "data" / "interim" / "expedia_bookings_no_tgt.csv"
if not BK_CSV.exists():
    subprocess.run([str(ROOT / ".venv" / "Scripts" / "python.exe"), "-c",
                    "import pandas as pd, sys; d = pd.read_parquet(sys.argv[1]); "
                    "d = d[[c for c in d.columns if not c.startswith('tgt_') and c != 'aux_cnt']]; d.to_csv(sys.argv[2], index=False)",
                    str(ROOT / "data" / "interim" / "expedia_bookings.parquet"), str(BK_CSV)], check=True)
bk = pd.read_csv(BK_CSV, low_memory=False)
assert len(bk) == n_events and (bk["user_id"].to_numpy() == data.user).all() and (bk["item_id"].to_numpy() == data.item).all()
ctx = [c for c in bk.columns if c.startswith("ctx_")] + [c for c in bk.columns if c.startswith("flag_")]
base = bk[["user_id", "item_id", "timestamp"]].copy()
base["event_id"] = np.arange(n_events)
base["n_prior"] = data.n_prior
base["is_cold"] = data.n_prior == 0
base = pd.concat([base, bk[ctx]], axis=1)
base["split"] = split
print(base.shape, "| leak columns kept:", [c for c in base.columns if c.startswith("tgt_") or c == "aux_cnt"])

In [ ]:
@torch.no_grad()
def predict(events: np.ndarray) -> pd.DataFrame:
    """Batched Runner.recommend: full ranking of the 100 clusters for each event id.

    Input: events (n,) event ids. Output: one row per event with top-10 clusters (raw ids),
    rank of the booked cluster, known-destination flag and sameDest weight.
    """
    d, r = data, runner
    rows = []
    for s in range(0, len(events), BATCH):
        ev_b = events[s : s + BATCH]
        n = len(ev_b)
        seq = np.zeros((n, r.s_len), np.int64)
        h_dest = np.zeros((n, r.s_len), np.int64)
        length = np.zeros(n, np.int64)
        for i, e in enumerate(ev_b):
            a, _ = d.u_start[int(d.user[e])]
            h = d.by_user[a : a + d.n_prior[e]][-r.s_len :]  # bookings of the user before e
            length[i] = len(h)
            seq[i, : len(h)] = [d.cluster2id[int(d.item[x])] for x in h]
            h_dest[i, : len(h)] = d.codes[h, 0]
        codes = d.codes[ev_b].astype(np.int32)
        q_idx = torch.from_numpy(encode(codes, r.vocab)).long()
        sc = r.model.scores(torch.from_numpy(seq), q_idx)
        sc[:, 0] = float("-inf")
        p = torch.softmax(sc, dim=1).numpy()[:, d.cluster_ids]
        q_dest = codes[:, 0].astype(np.int64)
        sd = same_dest_scores(seq, length, h_dest, q_dest, r.n_items, base=demo.RECENCY, normalize=True)[:, d.cluster_ids]
        known = ((h_dest == q_dest[:, None]) & (np.arange(r.s_len)[None] < length[:, None])).any(1)
        ws = np.where(known, r.ws[1], r.ws[0])
        score = smoothed_log(p, r.alpha) + ws[:, None] * smoothed_log(same_dest_distribution(sd), r.alpha)
        order = np.argsort(-score, axis=1, kind="stable")
        truth = d.item[ev_b]
        rows.append(pd.DataFrame({
            "event_id": ev_b,
            "top10_clusters": [" ".join(map(str, o[:TOP_K])) for o in order],
            "truth_rank": (order == truth[:, None]).argmax(1) + 1,
            "known_destination": known,
            "w_samedest": ws,
        }))
    return pd.concat(rows, ignore_index=True)


order_pop = np.argsort(-baselines.pop, kind="stable")
field_to_code_idx = {f: i for i, f in enumerate(demo.QUERY_FIELDS)}
cat_maps = []
for i, f in enumerate(demo.LR_ORDER):
    cat_list = baselines.cats[i]
    cat_to_col = {cat: baselines.offset[i] + idx for idx, cat in enumerate(cat_list)}
    cat_maps.append(cat_to_col)
coef_T_padded = np.vstack([baselines.coef.T, np.zeros((1, demo.N_CLUSTERS), dtype=baselines.coef.dtype)])


def predict_baselines(events: np.ndarray, batch_size: int = 16384) -> pd.DataFrame:
    """Vectorized batched ItemKNN and Logistic Regression top-10 predictions."""
    n = len(events)
    knn_top10, lr_top10 = [], []
    for start in range(0, n, batch_size):
        ev_b = events[start : start + batch_size]
        nb = len(ev_b)

        # 1. ItemKNN
        H = np.zeros((nb, demo.N_CLUSTERS), dtype=np.float32)
        has_hist = np.zeros(nb, dtype=bool)
        for i, e in enumerate(ev_b):
            a, _ = data.u_start[int(data.user[e])]
            h = data.by_user[a : a + data.n_prior[e]][-20:]
            if len(h) > 0:
                has_hist[i] = True
                np.add.at(H[i], data.item[h], 1)
        knn_scores = H @ baselines.sim
        if not has_hist.all():
            knn_scores[~has_hist] = baselines.pop_cold
        knn_perm = knn_scores[:, order_pop]
        knn_sorted = np.argsort(-knn_perm, axis=1, kind="stable")
        knn_top = order_pop[knn_sorted[:, :TOP_K]]
        knn_top10.extend([" ".join(map(str, row)) for row in knn_top])

        # 2. Logistic Regression
        codes_b = data.codes[ev_b]
        col_matrix = np.full((nb, len(demo.LR_ORDER)), -1, dtype=np.int32)
        for i, f in enumerate(demo.LR_ORDER):
            c_idx = field_to_code_idx[f]
            raw_vals = codes_b[:, c_idx]
            cmap = cat_maps[i]
            if f in ("ci_month", "adults", "rooms"):
                vals = np.where(raw_vals > 0, raw_vals, -1)
            elif f in ("stay", "lead"):
                vals = np.where(raw_vals > 0, raw_vals - 1, -1)
            else:
                vals = raw_vals
            col_matrix[:, i] = [cmap.get(str(v), -1) for v in vals]
        lr_scores = coef_T_padded[col_matrix].sum(axis=1) + baselines.icpt
        lr_perm = lr_scores[:, order_pop]
        lr_sorted = np.argsort(-lr_perm, axis=1, kind="stable")
        lr_top = order_pop[lr_sorted[:, :TOP_K]]
        lr_top10.extend([" ".join(map(str, row)) for row in lr_top])

    return pd.DataFrame({
        "event_id": events,
        "top10_clusters_itemknn": knn_top10,
        "top10_clusters_lr": lr_top10,
    })


# check against the single-row demo code
rng = np.random.default_rng(0)
chk = np.sort(rng.choice(np.arange(bounds["test"][0], n_events), 40, replace=False))
got = predict(chk)
got_bl = predict_baselines(chk)
for e, row, row_bl in zip(chk, got.itertuples(), got_bl.itertuples()):
    fields = {f: int(c) for f, c in zip(QUERY_FIELDS, data.codes[e])}
    ref = runner.recommend(data.history(int(data.user[e]), int(e)), fields, int(data.item[e]))
    ref_bl = baselines.recommend([int(data.item[x]) for x in data.history(int(data.user[e]), int(e))], fields, int(data.item[e]))
    assert [t["cluster"] for t in ref["top"]] == list(map(int, row.top10_clusters.split()))[:5], e
    assert ref["truth"]["rank"] == row.truth_rank, e
    assert ref_bl["ItemKNN"]["top"] == list(map(int, row_bl.top10_clusters_itemknn.split()))[:5], e
    assert ref_bl["Logistic regression"]["top"] == list(map(int, row_bl.top10_clusters_lr.split()))[:5], e
print("batched top-5 and truth rank equal Runner.recommend and Baselines.recommend on", len(chk), "test rows")


In [ ]:
test_events = np.flatnonzero(split == "test")
pred = predict(test_events)
pred_bl = predict_baselines(test_events)
test = base[base["split"] == "test"].merge(pred, on="event_id", how="left", validate="1:1")
test = test.merge(pred_bl, on="event_id", how="left", validate="1:1")
assert test["top10_clusters"].notna().all() and test["top10_clusters_itemknn"].notna().all() and test["top10_clusters_lr"].notna().all() and len(test) == len(test_events)
test = test.drop(columns="split")
base[base["split"] == "train"].drop(columns="split").to_csv(OUT / "train.csv", index=False)
valid_events = np.flatnonzero(split == "valid")
valid = base[base["split"] == "valid"].merge(predict(valid_events), on="event_id", how="left", validate="1:1")
valid = valid.merge(pred_bl_valid := predict_baselines(valid_events), on="event_id", how="left", validate="1:1")
assert valid["top10_clusters"].notna().all() and valid["top10_clusters_itemknn"].notna().all() and valid["top10_clusters_lr"].notna().all() and len(valid) == len(valid_events)
valid = valid.drop(columns="split")
valid_hit = valid["truth_rank"] <= TOP_K

top10_hit = test["truth_rank"] <= TOP_K
warm = ~test["is_cold"]
counts = {
    "model": "SMLP4Rec + query embedding (01d, first bookings in train) + sameDest", "seed": SEED, "dataset": "Expedia, 100 clusters",
    "boundaries_first_target_event": {k: v[0] for k, v in bounds.items()},
    "rows": {k: int((split == k).sum()) for k in ("train", "valid", "test")},
    "test_warm_rows": int(warm.sum()), "test_cold_rows": int((~warm).sum()),
    "valid_recall_at_10_warm": float(valid_hit[~valid["is_cold"]].mean()), "valid_recall_at_10_cold": float(valid_hit[valid["is_cold"]].mean()),
    "test_recall_at_10_warm": float(top10_hit[warm].mean()), "test_recall_at_10_cold": float(top10_hit[~warm].mean()),
}
valid.drop(columns=["truth_rank", "known_destination", "w_samedest"]).to_csv(OUT / "valid.csv", index=False)
test.drop(columns=["truth_rank", "known_destination", "w_samedest"]).to_csv(OUT / "test.csv", index=False)  # top-10 clusters added
COUNTS_JSON.write_text(json.dumps(counts, indent=1), encoding="utf-8")
print(json.dumps(counts, indent=1))
test.head()
